# 2.5 — does focal loss add anything on top of rotation?

Set `SERIES` below. Two series exist, and the second supersedes the first:

- **`v26_focal_cnn`** — on `baseline_cnn`, the model that actually wins. **Run this one.**
- `v25_focal` — the same question on `baseline_v2`, kept for the record.

| arm | loss | gamma | sampler | expectation |
|---|---|---|---|---|
| `sampler` (control) | cross-entropy | — | inverse-sqrt | the 0.8800 arm from phase 2 |
| `focal_g2-sampler` | focal | 2.0 | inverse-sqrt | tied its control on `baseline_v2` |
| `focal_g1-sampler` | focal | 1.0 | inverse-sqrt | least likely to fight the sampler |
| `focal_g2` | focal | 2.0 | none | expected to lose |
| `focal_g1` | focal | 1.0 | none | expected to lose |

## Why v25 was rerun on a different model

`baseline_v2` was built to fix overfitting — 84% of `baseline_cnn`'s parameters sit in one
1024->128 layer, and global pooling cut 157k to 34k. But rotation had already removed the
overfitting on its own, so that cut only removed capacity. Measured: all four v2 arms
plateaued at 0.78-0.80 after 50 epochs while `baseline_cnn` reached ~0.87. **Drop v2.**

## Two settings mistakes this series avoids

**Defaults inheritance.** Moving configs into a subdirectory broke it silently: the loader
looked for `defaults.yaml` in the immediate parent only, found none, and merged nothing. The
first v25 pass therefore trained at batch 512 / lr 1e-3 / 40 epochs / patience 5. The loader
now walks upward and warns when there is none, and the run cell asserts the values.

**Patience.** 7 was chosen by replaying one recorded metric series where it sufficed. Runs
are not deterministic, and this arm plateaus before climbing late: a rerun at patience 7
stopped at epoch 25, while the patience-10 run of the same setup found its best at epoch 36
(0.8800). Back to 10.

## What already holds

Focal **with** the sampler tied the control on `baseline_v2` (0.7828 vs 0.7830); focal
**without** it lost badly (~0.70). The sampler makes rare classes common in a batch and
focal then down-weights exactly those once the model gets them right — so they can cancel
rather than compound. This series asks whether that conclusion survives on the better model.

Judge against the control, and treat a gain under ~0.02 as unconfirmed: validation has been
selected on heavily enough that roughly that much optimism is priced in.

## 1. Setup

Same bootstrap as the phase-2 notebook; every step is a no-op if already done.

In [1]:
import os, shutil, subprocess, sys
from pathlib import Path


def looks_like_the_repository(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "fdl_project").is_dir()


REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/fdl-project")]
     if looks_like_the_repository(p)),
    None,
)
assert REPO is not None, "clone the repo to /content/fdl-project first"
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                "-e", str(REPO), "--no-deps"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)

import torch

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE = DRIVE_ROOT / "BICOCCA/FDL"
DATASET = REPO / "data/MIR-WM811K/WM811K.pkl"
EXPECTED_BYTES = 2_022_961_642


def mount_drive() -> bool:
    if DRIVE_ROOT.is_dir():
        return True
    try:
        from google.colab import drive

        drive.mount("/content/drive")   # idempotent; never force_remount
    except Exception as error:
        print(f"  Drive unavailable ({type(error).__name__})")
        return False
    return DRIVE_ROOT.is_dir()


HAS_DRIVE = mount_drive()
if not DATASET.exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE / "DATA/data/MIR-WM811K/WM811K.pkl", DATASET)
assert DATASET.stat().st_size == EXPECTED_BYTES, "wrong pickle: splits are row indices"

CHECKPOINTS = DRIVE / "checkpoints"
if HAS_DRIVE:
    CHECKPOINTS.mkdir(parents=True, exist_ok=True)

print(f"gpu     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"drive   {'mounted' if HAS_DRIVE else 'NOT mounted'}")
print(f"dataset {DATASET.stat().st_size / 1024**3:.2f} GiB")

gpu     NVIDIA L4
drive   mounted
dataset 1.88 GiB


## 2. W&B

`wandb login` in the terminal is the reliable path — Colab Secrets time out when the runtime
is driven from VS Code. `~/.netrc` is read by both this kernel and any terminal process.

In [2]:
USE_WANDB = True
WANDB_PROJECT = "wm811k-wafer-defects"

if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb

    if not wandb.api.api_key:
        USE_WANDB = False
        print("  not authenticated -- run `wandb login` in the terminal, then rerun")
    else:
        print(f"  wandb ready, project {WANDB_PROJECT!r}")

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


  wandb ready, project 'wm811k-wafer-defects'


## 3. Run the five arms

Sequentially, on one loaded copy of the source table. `transform_device=cuda` is on: rotation
becomes a single batched `grid_sample` and the encoding happens after the transfer, which is
worth the most on precisely this augmentation.

Each arm writes its own artifacts and appends to the results list as it finishes, so an
interrupted session keeps whatever completed.

In [3]:
import time

import pandas as pd

from fdl_project.config.loader import load_experiment_config
from fdl_project.data.datasets import load_wm811k_dataframe
from fdl_project.training.runner import run_experiment

SERIES = "v26_focal_cnn"          # or "v25_focal" for the baseline_v2 record
CONFIGS = sorted((REPO / "configs/train" / SERIES).glob("*.yaml"))
OVERRIDES = ["data.transform_device=cuda"]
if HAS_DRIVE:
    OVERRIDES.append(f"checkpoint.directory={CHECKPOINTS}")
if USE_WANDB:
    OVERRIDES += ["logging.wandb.enabled=true",
                  f"logging.wandb.project={WANDB_PROJECT}"]

dataframe = load_wm811k_dataframe(DATASET)
results = []

for path in CONFIGS:
    config = load_experiment_config(path, overrides=OVERRIDES)
    # Guard against the bug that invalidated the first pass: these values come
    # from configs/train/defaults.yaml, so if inheritance breaks again the run
    # stops here instead of quietly training on dataclass defaults.
    assert config.trainer.max_epochs == 50, "defaults.yaml was not inherited"
    assert config.trainer.batch_size == 256, "defaults.yaml was not inherited"

    print(f"\n=== {config.name}  ({config.model.name}, {config.imbalance.loss}, "
          f"gamma {config.imbalance.focal_gamma}, sampling {config.imbalance.sampling})")
    started = time.monotonic()
    result = run_experiment(config, overwrite=True, dataframe=dataframe)
    macro = result.bootstrap.aggregate.set_index("metric").loc["macro_f1"]
    results.append({
        "run": config.name,
        "model": config.model.name,
        "loss": config.imbalance.loss,
        "gamma": config.imbalance.focal_gamma if config.imbalance.loss == "focal" else None,
        "sampler": config.imbalance.sampling,
        "macro_f1": round(float(macro.point_estimate), 4),
        "ci_lower": round(float(macro.ci_lower), 4),
        "ci_upper": round(float(macro.ci_upper), 4),
        "best_epoch": result.fit.best_epoch,
        "epochs": len(result.fit.history),
        "minutes": round((time.monotonic() - started) / 60, 1),
    })
    row = results[-1]
    flag = "  <-- still improving at the cap" if row["best_epoch"] >= row["epochs"] - 2 else ""
    print(f"    macro-F1 {macro.point_estimate:.4f} "
          f"[{macro.ci_lower:.4f}, {macro.ci_upper:.4f}]  "
          f"best {row['best_epoch']}/{row['epochs']}  {row['minutes']:.1f} min{flag}")


=== baseline_cnn-rotation-sampler  (baseline_cnn, cross_entropy, gamma 2.0, sampling weighted)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: vlad-yelisieiev-bicocca (vlad-yelisieiev-bicocca-milano-bicocca) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


epoch,▁▅█
epoch_seconds,█▁▁
learning_rate,▁▁▁
train_accuracy,▁▃█
train_loss,█▇▁
train_samples,▁▁▁
validation_accuracy,▁██
validation_balanced_accuracy,█▃▁
validation_f1_Center,▁▄█
validation_f1_Donut,█▁█
+10,...


    macro-F1 0.8696 [0.8538, 0.8822]  best 18/28  1.1 min

=== baseline_cnn-rotation-focal_g1  (baseline_cnn, focal, gamma 1.0, sampling shuffle)


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
epoch_seconds,█▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▃▂▁▂▂▂▂▂▂▂▁▂▂▂▂▂▂▂
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▁▄▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇███████████████
train_loss,█▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_samples,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
validation_accuracy,▁▄▅▅▅▆▆▅▆▄▆▆▆▇▅▇▆▇▇▇▄▇▇▇▇▆█▇▇▇▇██▇▇████▇
validation_balanced_accuracy,▁▃▄▄▄▅▅▆▄▅▅▅▅▆▄▇▆▆▅▆▆▇▇▇▇▅▇▇█▆▇▆▆▇▇▇▇▇█▇
validation_f1_Center,▁▆▅▆▆▆▇▃▇▄▇▇▇▇▃▇▄▇██▆▆▇▇▇▅█▆█▆▇█▇█▇▇▇█▇▇
validation_f1_Donut,▁▂▆▅▅▅▆▅▅▆▆▆▅▅▅▄▆▇▅▆▆▆▆▆▇▇▆▇█▆▆▆▆▇▇▅▆▆▆▄
+10,...


...

    macro-F1 0.8738 [0.8593, 0.8866]  best 20/30  2.6 min

=== baseline_cnn-rotation-focal_g2  (baseline_cnn, focal, gamma 2.0, sampling shuffle)


epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
epoch_seconds,█▂▂▂▂▂▁▂▂▂▁▂▂▂▂▁▂▁▂▂▁▂▂▁▂▁▂▁▂▁▁▂▂▂▁▁▁▂▁▁
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▁▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇████████████████
train_loss,█▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_samples,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
validation_accuracy,▁▄▅▅▆▆▆▇▃▆▇▇▇▃▇▆▇▇▇▇▇▇▇█▇▇▇█▇▇▇▇██▇███▇▇
validation_balanced_accuracy,▁▅▅▅▅▅▅▆▇▆▆▆▆▄▆▇▇▆▆▇▇▆▇▇▇▇█▇▆█▇▇▇▇▇▇▇██▇
validation_f1_Center,▁▅▅▇▇▇▇▇▂▆▇█▇▁▇▅▆▇▇▆▇▇▇▇▇▇▅▇▇▇▇▇▇▇▇▇▆▇▇▆
validation_f1_Donut,▁▄▅▃▅▆▇▅▄▇▆█▆▆▆▄▇▆▇▇█▆▆▇▆▇▇▄▆▇█▆▆▅▇▆▇▇▇▅
+10,...


    macro-F1 0.8600 [0.8450, 0.8739]  best 47/50  3.9 min

=== baseline_cnn-rotation-focal_g2-sampler  (baseline_cnn, focal, gamma 2.0, sampling weighted)


epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
epoch_seconds,█▂▂▂▂▂▂▂▃▂▁▂▁▁▂▂▂▂▂▂▂▂▂▂▁▂▁▂▂▂▂▂▂▁▂▂▁▂▁▂
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▁▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇████████████████████
train_loss,█▄▄▃▃▃▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_samples,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
validation_accuracy,▆▇▆▁▅▅▇▇▅▇██▅▇▇█▇▇▇██▇▇▇▇▆▇███▇███▇▇▇█▇█
validation_balanced_accuracy,▁▁▄▆▅▆▆▆▇▆▅▄▇▆▇▇▇▇▇▆▆█▇▇██▇▇▇▆██▇▇█▇▇▆█▇
validation_f1_Center,▅▆▁▃▂▂▆▂▁▅██▂▄▅▇▄▅▆▇▆▅▅▆▅▅▅▇▆▇▅▇▇▆▆▄▇█▄▆
validation_f1_Donut,▆▁▆▃▃▅▇▆▄▆█▆▅▆▅▇▇▇▅▅▇▅▆▄▄▅▆█▅▆▆▅▇▇▆▇█▇▇▇
+10,...


    macro-F1 0.8711 [0.8554, 0.8839]  best 30/40  3.3 min


## 4. Read the result

`clears_control` is the test that matters: an arm counts only if its interval sits entirely
above the control's point estimate. With nine classes and 30 `Near-full` wafers in
validation, point estimates alone are not a ranking.

In [4]:
frame = pd.DataFrame(results)
control = frame.loc[frame["run"].str.endswith("-rotation-sampler"), "macro_f1"].squeeze()
frame["vs_control"] = (frame["macro_f1"] - control).round(4)
frame["clears_control"] = frame["ci_lower"] > control
# A best epoch at the cap means the run was still improving when it stopped:
# that number is a floor, not a result.
frame["truncated"] = frame["best_epoch"] >= frame["epochs"] - 2

pd.set_option("display.width", 220)
display(frame.sort_values("macro_f1", ascending=False))

if frame["truncated"].any():
    print("\nTruncated (still improving at the cap):",
          ", ".join(frame.loc[frame["truncated"], "run"]))
    print("Raise trainer.max_epochs before drawing conclusions from those.")

print(f"\nControl {float(control):.4f}. Phase 2 scored the same setup at 0.8800.")

winners = frame[frame["clears_control"] & ~frame["run"].str.endswith("rotation-sampler")]
if winners.empty:
    print("\nNothing clears the control. Imbalance handling is not the binding "
          "constraint here either -- report that as the finding.")
else:
    print("\nClears the control:", ", ".join(winners["run"]))
    print("Treat gains under ~0.02 as unconfirmed until the test split is opened.")

,run,model,loss,gamma,sampler,macro_f1,ci_lower,ci_upper,best_epoch,epochs,minutes,vs_control,clears_control,truncated
2,baseline_cnn-rotation-focal_g1-sampler,baseline_cnn,focal,1.0,weighted,0.8738,0.8593,0.8866,20,30,2.6,0.0042,False,False
4,baseline_cnn-rotation-focal_g2-sampler,baseline_cnn,focal,2.0,weighted,0.8711,0.8554,0.8839,30,40,3.3,0.0015,False,False
0,baseline_cnn-rotation-sampler,baseline_cnn,cross_entropy,NaN,weighted,0.8696,0.8538,0.8822,18,28,1.1,0.0000,False,False
1,baseline_cnn-rotation-focal_g1,baseline_cnn,focal,1.0,shuffle,0.8624,0.8462,0.8760,48,50,3.9,-0.0072,False,True
3,baseline_cnn-rotation-focal_g2,baseline_cnn,focal,2.0,shuffle,0.8600,0.8450,0.8739,47,50,3.9,-0.0096,False,False



Truncated (still improving at the cap): baseline_cnn-rotation-focal_g1
Raise trainer.max_epochs before drawing conclusions from those.

Control 0.8696. Phase 2 scored the same setup at 0.8800.

Nothing clears the control. Imbalance handling is not the binding constraint here either -- report that as the finding.


In [5]:
OUTPUT = REPO / "output" / SERIES
OUTPUT.mkdir(parents=True, exist_ok=True)
frame.to_csv(OUTPUT / "results.csv", index=False)
if HAS_DRIVE:
    # Local disk dies with the session; Drive does not.
    shutil.copy2(OUTPUT / "results.csv", DRIVE / f"{SERIES}_results.csv")
    print("copied to", DRIVE / f"{SERIES}_results.csv")
print(OUTPUT / "results.csv")

copied to /content/drive/MyDrive/BICOCCA/FDL/v26_focal_cnn_results.csv
/content/fdl-project/output/v26_focal_cnn/results.csv


## 5. What follows

If an arm clears the control, it becomes part of the pipeline and the next series runs on
top of it. If nothing does — the likelier outcome given that every imbalance arm in phase 2
landed inside ±0.013 — then the finding is that **augmentation is the only lever that
mattered on this dataset**, and the report says so with the numbers behind it.

Either way the next step is the same: three seeds on whichever setup wins, to separate a
real difference from selection on a noisy validation metric.